In [8]:
import sys
sys.path.append('../utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer
import os
from dotenv import load_dotenv
import json
import random

In [4]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

In [5]:
orig_df = pd.read_csv('../data/initial_datasets/dota2/dota2_grouped.csv')

In [6]:
orig_df = orig_df.rename(columns={"translated_message": "sentences"})
orig_df

,match,player,sentences,label
0,0,2,"""I'm sure"". ""that there's a ward here"". ""Am I ...",0
1,0,4,"""Two-step move SK"". ""won't work"". ""how did you...",0
2,0,6,"""high ground""",0
3,0,7,"""Place a sentry."". ""you'll find out"". ""Place a...",0
4,0,9,"""weakest invoker"". ""like the tinker""",0
...,...,...,...,...
580,99,0,gg wp,0
581,99,1,ggwp,0
582,99,2,GG,0
583,99,6,"The message is already in English.. ""saw me te...",0


In [ ]:
prompt = 'You are generating a small, labeled dataset of realistic in-game chat logs from a single player in a DOTA 2 match.  ' \
'\n\nOutput exactly 10 examples, each on its own line, in valid JSON-lines (one JSON object per line).  \nEach object must have ' \
'two keys:  \n• "chat": a single string that concatenates **all** the player’s messages from that match, separated by a period (“.”). ' \
'End the last message with a period as well.  \n• "label": 1 for non-toxic players, ‑1 for toxic players.  \n\nContent requirements  \n1. ' \
'Create 5 toxic (label ‑1) and 5 non-toxic (label 1) examples.  \n2. Make the chats feel authentic to DOTA 2: mention hero names (“Invoker”, ' \
'“Pudge”, “Crystal Maiden”, etc.), roles (“pos 5”, “mid”, “offlane”), items (“Blink”, “BKB”), map objectives (“Roshan”, “T1 mid”), slang (“gg”, ' \
'“glhf”, “ward pls”) and timing (“20 min”, “pre-horn”).  \n3. Toxic chats (-1) should include frustration, flaming, caps, sarcasm, profanity (e.g., ' \
'“shit”, “bitch”, “fuck”), or personal attacks, but **no hate speech, slurs, or sexual content**.  \n4. Non-toxic chats (1) should be polite, constructive, ' \
'or encouraging.  \n5. Keep each “chat” between 40 – 120 words total.  \n6. Do not add extra fields, line breaks, or commentary—only the 10 JSON objects.  ' \
'\n\nExample format (do NOT output this example in your final answer):  \n{"chat":"glhf team. remember to ward rune. I will go mid Invoker. call missing pls.",' \
'"label":1}\n\nNow produce the 10 JSON-line examples following all rules above.'

In [7]:
prompt = f"Simulate realistic game text chat exchanges (using abbreviations, typos due to typing fast during the game\
, COD slang, fustrated repeats when banging on the same key, etc.) \
between player A and Player B in COD text game chat. Communication from Player A is characterized as [DEFA]\
Player B is characterized as [DEFB]. \
Format as [P1] msg1 \n [P2] msg2 \n ..."

In [9]:
print(prompt)

You are generating a small, labeled dataset of realistic in-game chat logs from a single player in a DOTA 2 match.  

Output exactly 10 examples, each on its own line, in valid JSON-lines (one JSON object per line).  
Each object must have two keys:  
• "chat": a single string that concatenates **all** the player’s messages from that match, separated by a period (“.”). End the last message with a period as well.  
• "label": 1 for non-toxic players, ‑1 for toxic players.  

Content requirements  
1. Create 5 toxic (label ‑1) and 5 non-toxic (label 1) examples.  
2. Make the chats feel authentic to DOTA 2: mention hero names (“Invoker”, “Pudge”, “Crystal Maiden”, etc.), roles (“pos 5”, “mid”, “offlane”), items (“Blink”, “BKB”), map objectives (“Roshan”, “T1 mid”), slang (“gg”, “glhf”, “ward pls”) and timing (“20 min”, “pre-horn”).  
3. Toxic chats (-1) should include frustration, flaming, caps, sarcasm, mild profanity (e.g., “damn”, “crap”, “suck”), or personal attacks, but **no hate sp

In [11]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [17]:
res = []
data = {
    'translated_messages': [],
    'label': []
}
for i in tqdm(range(125)):

    prompt = f"Simulate realistic game text chat exchanges (using abbreviations, typos due to typing fast during the game\
    , COD slang, fustrated repeats when banging on the same key, etc.) \
    between player A and Player B in COD text game chat. Communication from Player A is characterized as [DEFA]\
    Player B is characterized as [DEFB]. \
    Format as [P1] msg1 \n [P2] msg2 \n ..."

    defa = random.randint(0, 1)
    defb = random.randint(0, 1)
    if defa == 0:
        defa = "toxic"
    else:
        defa = "non-toxic"
    
    if defb == 0:
        defb = "toxic"
    else:
        defb = "non-toxic"
        

    prompt = f"{prompt.replace('[DEFA]', defa).replace('[DEFB]', defb)}"

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    resp = response.choices[0].message.content


    p1_lines = []
    p2_lines = []

    for line in resp.splitlines():
        line = line.strip()
        if line.startswith("[P1]"):
            p1_lines.append(f"\"{line[5:]}\"")
        elif line.startswith("[P2]"):
            p2_lines.append(f"\"{line[5:]}\"")


    data['translated_messages'].append('. '.join(p1_lines))
    data['label'].append(defa)

    data['translated_messages'].append('. '.join(p2_lines))
    data['label'].append(defb)

100%|██████████| 125/125 [15:26<00:00,  7.42s/it]


In [ ]:
res = []
for i in range(100):
    response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input
    )
    res.append(response.output_text)

In [ ]:
# instruction = (
#     "You are a data generator tasked with creating realistic DOTA 2 chat messages from a single player. "
#     "These chat messages should be labeled according to their sentiment: toxic or non-toxic.\n"
#     "Base the style on typical video game chat messages — include informal internet language, typos, and abbreviations\n"
#     "Generate exactly 10 realistic DOTA 2 chat messages for each single player, with each message the player said followed by a bullet point\n"
#     "Each line should follow this format: the chat message in double quotes, followed by a space and then the label (0 for toxic, 1 for non-toxic).\n"
#     "No extra formatting — just plain text output, one line per comment.\n"
#     "Here is the format:\n"
#     "\"gg dawg\" 0\n"
#     "\"I hate u bitch\" 1"
# )
# input = (
#     f"Now, generate the 10 new comments below:"
# )

In [27]:
json.loads(res[0].split("\n")[0])

{'chat': "Hey team, let's have a great game. I'll go offlane Pudge. Remember to ward and call missing. Good luck!",
 'label': 1}

In [18]:
df = pd.DataFrame(data)
df

,translated_messages,label
0,"""hey! u ready?"". ""drop at farm?"". ""watch tower...",non-toxic
1,"""yep let's do this!"". ""farm is hot! but ok"". ""...",non-toxic
2,"""glhf team!"". ""haha let's do our best!"". ""srry...",non-toxic
3,"""omfg not another noob..."". ""bruh u cant even ...",toxic
4,"""omggg u suck, cant u aim???"". ""ur trash bro l...",toxic
...,...,...
245,"""lol git gud kid"". ""nah ur just trash, uninsta...",toxic
246,"""GLHF! Let's get this win!"". ""Doing my best, g...",non-toxic
247,"""u better not b trash, noob"". ""pffft ur prob g...",toxic
248,"""glhf team!"". ""lol ok, where we drop?"". ""XD le...",non-toxic


In [19]:
df['label'] = df['label'].replace({'non-toxic': 0, 'toxic': 1})
generated_df = df

/var/folders/lv/pnwq6bmj4tq68bsvy__37qyh0000gn/T/ipykernel_9673/537067671.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['label'] = df['label'].replace({'non-toxic': 0, 'toxic': 1})


In [28]:
dicts = []
for s in res:
    texts = s.split("\n")
    for txt in texts:
        dicts.append(json.loads(txt))

generated_df = pd.DataFrame(dicts)

In [20]:
generated_df

,translated_messages,label
0,"""hey! u ready?"". ""drop at farm?"". ""watch tower...",0
1,"""yep let's do this!"". ""farm is hot! but ok"". ""...",0
2,"""glhf team!"". ""haha let's do our best!"". ""srry...",0
3,"""omfg not another noob..."". ""bruh u cant even ...",1
4,"""omggg u suck, cant u aim???"". ""ur trash bro l...",1
...,...,...
245,"""lol git gud kid"". ""nah ur just trash, uninsta...",1
246,"""GLHF! Let's get this win!"". ""Doing my best, g...",0
247,"""u better not b trash, noob"". ""pffft ur prob g...",1
248,"""glhf team!"". ""lol ok, where we drop?"". ""XD le...",0


In [21]:
generated_df.to_csv('../data/generated/dota2/control/grouped_control_gen_fixed.csv', index=False)